M1_Data_Pipeline_OSA
code M1

In [1]:
import numpy as np
import os
import torch
import random
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

drive.mount('/content/drive')

# Chemins
RAW_PATH = '/content/drive/MyDrive/apnea-ecg-database-1.0.0/final_dataset'
CLEAN_PATH = '/content/drive/MyDrive/apnea-ecg-database-1.0.0/clean_dataset'

if not os.path.exists(CLEAN_PATH):
    os.makedirs(CLEAN_PATH)
    print(f"📁 Nouveau dossier créé : {CLEAN_PATH}")

Mounted at /content/drive
📁 Nouveau dossier créé : /content/drive/MyDrive/apnea-ecg-database-1.0.0/clean_dataset


check_data.py

In [2]:
def synchroniser_et_sauvegarder():
    for s in ['train', 'val', 'test']:
        # Chargement des fichiers bruts
        x_ecg = np.load(os.path.join(RAW_PATH, f'X_ecg_{s}.npy'))
        x_hrv = np.load(os.path.join(RAW_PATH, f'X_hrv_{s}.npy'))
        y = np.load(os.path.join(RAW_PATH, f'y_{s}.npy'))

        # Calcul de la taille commune (Synchronisation)
        min_s = min(len(x_ecg), len(x_hrv), len(y))

        # Sauvegarde physique des nouveaux fichiers "CLEAN"
        np.save(os.path.join(CLEAN_PATH, f'X_ecg_{s}_clean.npy'), x_ecg[:min_s])
        np.save(os.path.join(CLEAN_PATH, f'X_hrv_{s}_clean.npy'), x_hrv[:min_s])
        np.save(os.path.join(CLEAN_PATH, f'y_{s}_clean.npy'), y[:min_s])

        print(f"✅ {s.upper()} : Sauvegardé avec {min_s} lignes dans /clean_dataset/")

# On lance la machine
synchroniser_et_sauvegarder()

✅ TRAIN : Sauvegardé avec 33907 lignes dans /clean_dataset/
✅ VAL : Sauvegardé avec 4638 lignes dans /clean_dataset/
✅ TEST : Sauvegardé avec 5827 lignes dans /clean_dataset/


osa_dataset.py

In [10]:
class OSADataset(Dataset):
    def __init__(self, stage, seq_len=10, augment=False):
        # Chargement
        self.X_ecg = torch.FloatTensor(np.load(os.path.join(CLEAN_PATH, f'X_ecg_{stage}_clean.npy'))).view(-1, 1, 6000)
        self.X_hrv = torch.FloatTensor(np.load(os.path.join(CLEAN_PATH, f'X_hrv_{stage}_clean.npy')))
        self.y = torch.LongTensor(np.load(os.path.join(CLEAN_PATH, f'y_{stage}_clean.npy')))

        self.seq_len = seq_len
        self.augment = augment
        print(f"📦 Dataset {stage} chargé. Augmentation: {self.augment}")

    def __len__(self):
        return len(self.y) - self.seq_len

    def __getitem__(self, idx):
        curr = idx + self.seq_len
        # On fait une COPIE (.clone()) pour ne pas modifier la base de données originale en RAM
        ecg = self.X_ecg[curr].clone()

        if self.augment:
            ecg = self._augment_ecg(ecg)

        return ecg, self.X_hrv[idx:curr], self.y[curr]

    def _augment_ecg(self, ecg_t):
        # On force un choix qui n'est PAS 'none' pour le test
        aug = random.choice(['noise', 'scale', 'shift'])
        sig = ecg_t.numpy().squeeze()

        if aug == 'noise':
            noise = np.random.normal(0, 0.01, sig.shape)
            sig = sig + noise
        elif aug == 'scale':
            sig = sig * np.random.uniform(0.8, 1.2)
        elif aug == 'shift':
            sig = np.roll(sig, np.random.randint(-100, 100))

        return torch.FloatTensor(sig).unsqueeze(0)

dataloaders.py

In [11]:
# Initialisation simple et propre
train_ds = OSADataset('train', augment=True)
val_ds   = OSADataset('val',   augment=False)
test_ds  = OSADataset('test',  augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

print(f"🚀 DataLoaders connectés aux fichiers CLEAN (Train: {len(train_loader)} batches)")

📦 Dataset train chargé. Augmentation: True
📦 Dataset val chargé. Augmentation: False
📦 Dataset test chargé. Augmentation: False
🚀 DataLoaders connectés aux fichiers CLEAN (Train: 1060 batches)


Test

In [12]:
print("🧪 TEST DE CONFORMITÉ DISQUE...")
ecg_b, hrv_b, lbl_b = next(iter(train_loader))

assert ecg_b.shape == (32, 1, 6000), "Erreur ECG"
assert hrv_b.shape == (32, 10, 8), "Erreur HRV"
print("✅ SUCCESS : Les fichiers sauvegardés sur le disque sont parfaits !")

🧪 TEST DE CONFORMITÉ DISQUE...
✅ SUCCESS : Les fichiers sauvegardés sur le disque sont parfaits !
